# <YOUR MODEL NAME> — Evaluation Tutorial

Interactive guide to running and developing the evaluation pipeline inside the Docker container.

## Start Jupyter

From `evaluation/models/<model_folder>/`:
```bash
docker compose --profile jupyter up jupyter
# Open: http://localhost:8888
```

All paths below are **container-internal**.

---
## 1. Inspect Data and Weights

In [ ]:
# List benchmark data
!ls /causal_drive_bench/

In [ ]:
# List raw dataset
!ls /raw_data/

In [ ]:
# List model weights
!ls /workspace/weights/

In [ ]:
# View a sample benchmark record
import json, pathlib

bench_dir = pathlib.Path("/causal_drive_bench/causal_nuscenes")
sample_dirs = sorted(bench_dir.glob("*_SAMPLED_*"))
print(f"Found {len(sample_dirs)} samples")

if sample_dirs:
    first = sample_dirs[0]
    print(f"\nFiles in {first.name}:")
    for f in sorted(first.iterdir()):
        print(f"  {f.name}")

In [ ]:
# View state.json for the first sample
state = json.loads((first / "state.json").read_text())
print(json.dumps(state, indent=2)[:2000])

In [ ]:
# View QA questions for the first sample
qa = json.loads((first / "qa.json").read_text())
for qa_type, questions in qa.items():
    print(f"\n--- {qa_type} ({len(questions)} questions) ---")
    if questions:
        print(json.dumps(questions[0], indent=2))

---
## 2. Visualise Sample Images

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

frames = json.loads((first / "frames.json").read_text())
# TODO: update key names to match your dataset structure
t0 = frames.get("Tp0p0", {})

cam_paths = {
    k: v for k, v in t0.items() if "path" in k and v
}

fig, axes = plt.subplots(1, len(cam_paths), figsize=(4 * len(cam_paths), 3))
if len(cam_paths) == 1:
    axes = [axes]
for ax, (name, path) in zip(axes, cam_paths.items()):
    try:
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(name, fontsize=8)
    except Exception as e:
        ax.set_title(f"{name}\n(missing)")
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 3. Check Current Outputs

In [ ]:
!ls /outputs/

In [ ]:
# Peek at existing outputs (if any)
import pathlib
output_files = list(pathlib.Path("/outputs").rglob("outputs.jsonl"))
print(f"Found {len(output_files)} outputs.jsonl file(s)")
if output_files:
    lines = output_files[0].read_text().strip().splitlines()
    print(f"\nFirst entry from {output_files[0]}:")
    print(json.dumps(json.loads(lines[0]), indent=2))

In [ ]:
# (Optional) Clear outputs — uncomment to reset
# import shutil
# for f in pathlib.Path("/outputs").rglob("outputs.jsonl"):
#     f.unlink()
# print("Outputs cleared")

---
## 4. Run Inference

In [ ]:
# Subset mode — fastest for development
# TODO: update path to match your model folder
!python /workspace/repo/evaluation/models/<model_folder>/inference.py \
    --mode subset --subset-size 1

In [ ]:
# Single scene — pinpoint debugging
# !python /workspace/repo/evaluation/models/<model_folder>/inference.py \
#     --mode single --scene nuscenes-scene-0001

In [ ]:
# Full dataset — production run
# !python /workspace/repo/evaluation/models/<model_folder>/inference.py

---
## 5. Run Post-processing

In [ ]:
# TODO: update path to match your model folder
!python /workspace/repo/evaluation/models/<model_folder>/postprocess.py \
    --mode subset --subset-size 1

In [ ]:
# View the report
import json, pathlib
report_path = pathlib.Path("/outputs/report.json")
if report_path.exists():
    print(json.dumps(json.loads(report_path.read_text()), indent=2))
else:
    print("No report.json found — run postprocess first")

---
## 6. Development Utilities

In [ ]:
# Inspect installed packages
!pip list | grep -E 'torch|transformers|pillow|numpy'

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"    Memory: {mem:.1f} GB")

In [ ]:
# Run tests from inside the notebook
!python -m pytest /workspace/repo/evaluation/models/<model_folder>/tests/ -v